# SciFact Evidence-Conditioned Claim Verification
## Reproducibility and Archived-Results Verification

**Student:** Umakant Singh  
**Enrollment:** MNS2026009  
**Course:** AI Tools for Research — IIIT Allahabad  
**Dataset:** SciFact

### Purpose

The repository already contains the implementation under `src/`, workflows under `scripts/`, tests under `tests/`, and archived outputs under `results/`. This notebook calls those existing scripts; it does not copy the implementation, invent results, or claim archived outputs are newly reproduced.

The primary comparison is A — claim only, B — claim plus annotated gold evidence, and C — claim plus BM25 top-five retrieved evidence. The archived comparison is on the matched SciFact development set, not the test set.

## 1. Set the project root
Run this notebook from the repository root.

In [ ]:
from pathlib import Path
import os, sys, subprocess, platform, json

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'scripts').is_dir() and (candidate / 'src' / 'scifact_verif').is_dir():
            return candidate
    raise FileNotFoundError('Open this notebook from inside the project repository.')

ROOT = find_project_root()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
print('Project root:', ROOT)
print('Python:', sys.version)
print('Platform:', platform.platform())

## 2. Check the repository structure
This cell checks for the existing project folders and scripts.

In [ ]:
required_paths = ['configs', 'data/README.md', 'notebooks', 'paper', 'results', 'scripts/02_audit_data.py', 'scripts/04_evaluate.py', 'scripts/05_run_retrieval.py', 'scripts/06_eval_retrieval.py', 'scripts/07_run_condition.py', 'scripts/14_verify_saved_outputs.py', 'scripts/15_audit_archived_results.py', 'src/scifact_verif', 'tests']
for item in required_paths:
    print(('FOUND' if (ROOT / item).exists() else 'MISSING'), item)

## 3. Check required Python packages
The dependency list is maintained in `requirements.lock.txt`. This cell checks imports without installing packages.

In [ ]:
import importlib.util
packages = ['numpy', 'pandas', 'sklearn', 'scipy', 'yaml', 'rank_bm25', 'matplotlib', 'pytest']
for package in packages:
    print(f"{package}: {'available' if importlib.util.find_spec(package) else 'NOT INSTALLED'}")

## 4. Verify archived prediction outputs
The existing read-only script checks prediction hashes, row counts, unique claim IDs, and matched claim IDs across A/B/C. It does not retrain models.

In [ ]:
command = [sys.executable, 'scripts/14_verify_saved_outputs.py']
result = subprocess.run(command, cwd=ROOT, text=True, capture_output=True)
print(result.stdout)
if result.stderr: print(result.stderr)
print('Exit code:', result.returncode)
if result.returncode != 0: raise RuntimeError('Archived prediction verification did not pass.')

## 5. Audit archived metrics and RQ3 sample structure
The existing audit recalculates accuracy and macro-F1 from saved predictions and checks the preliminary RQ3 sample structure. RQ3 codes still require student verification; this audit does not make them final human annotations.

In [ ]:
command = [sys.executable, 'scripts/15_audit_archived_results.py']
result = subprocess.run(command, cwd=ROOT, text=True, capture_output=True)
print(result.stdout)
if result.stderr: print(result.stderr)
print('Exit code:', result.returncode)
if result.returncode != 0: raise RuntimeError('Archived-results audit did not pass.')

## 6. Display archived primary development-set results
The table reads the saved metrics JSON. These are archived development-set results, not newly computed or test-set results.

In [ ]:
import pandas as pd
metrics_path = ROOT / 'results/metrics/eval_20261030T132835Z/metrics.json'
if not metrics_path.exists(): raise FileNotFoundError(f'Archived metrics file not found: {metrics_path}')
metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
rows = []
for condition in ['A', 'B', 'C']:
    result = metrics['results'][condition]['primary_matched']
    rows.append({'Condition': condition, 'Number of claims': result['n'], 'Accuracy': result['accuracy'], 'Macro-F1': result['macro_f1']})
display(pd.DataFrame(rows))

## 7. Display archived paper figures
The following cells display existing figure exports; they do not regenerate the plots.

In [ ]:
from IPython.display import display, Image
figure_files = [ROOT / 'paper/figures/macro_f1_by_condition.png', ROOT / 'paper/figures/paired_differences_gold_slice.png', ROOT / 'paper/figures/retrieval_recall_at5.png']
for figure_path in figure_files:
    if figure_path.exists():
        print(figure_path.relative_to(ROOT))
        display(Image(filename=str(figure_path)))
    else:
        print('Figure not found yet:', figure_path.relative_to(ROOT))

## 8. Optional: rerun the original data-dependent workflow
Do not run this section just to verify archived results. Raw SciFact files are excluded; follow `data/README.md` first. These commands use existing project scripts and create new timestamped outputs. The upstream `release/latest` data pointer is mutable, so byte-identical reproduction is not guaranteed.

### 8.1 Download and audit
```bash
bash scripts/01_download_data.sh
python scripts/02_audit_data.py --split dev --policy all_labeled --near-dup holdout --sensitivity-jaccard 0.7
```

### 8.2 Retrieval
```bash
python scripts/05_run_retrieval.py --matched-dir results/audit/dev_all_labeled_nd-holdout
```
Use the actual timestamped retrieval output directory as `RANKS_DIR`.

### 8.3 Conditions A, B, and C
```bash
python scripts/07_run_condition.py --condition A --matched-dir results/audit/dev_all_labeled_nd-holdout
python scripts/07_run_condition.py --condition B --matched-dir results/audit/dev_all_labeled_nd-holdout
python scripts/07_run_condition.py --condition C --matched-dir results/audit/dev_all_labeled_nd-holdout --ranks-dir RANKS_DIR
```
Replace `RANKS_DIR` with the actual retrieval directory. Each run creates a timestamped prediction directory.

### 8.4 Evaluate
```bash
python scripts/04_evaluate.py --pred A=PATH_TO_A_PREDICTIONS_CSV --pred B=PATH_TO_B_PREDICTIONS_CSV --pred C=PATH_TO_C_PREDICTIONS_CSV --matched-dir results/audit/dev_all_labeled_nd-holdout
python scripts/06_eval_retrieval.py --ranks-dir RANKS_DIR --matched-dir results/audit/dev_all_labeled_nd-holdout
```
Replace placeholders with actual paths. Do not describe archived timestamped outputs as newly generated.

## 9. Interpretation and limitations
- A, B, and C are the three evidence-input conditions in this project.
- The archived primary comparison uses 286 matched development claims.
- Retrieval Recall@5 is a separate retrieval metric; retrieval success does not establish correct claim classification.
- Saved-output verification checks file integrity and metric consistency; it does not prove an independent rerun.
- Raw SciFact data are not distributed in this repository.
- The full test suite has not been confirmed in the specified Python 3.11.2 environment.
- The 50 RQ3 rows have preliminary AI-assisted codes and require student verification; they are not final human annotations.

**Conclusion:** This notebook documents and verifies existing project artifacts. Report a new experiment as reproduced only after successfully running the data-dependent workflow in the intended environment.